# Feature/Target Separation

## Definition
Before training any ML model, you must separate:
- **Features (X)**: Input variables used to make predictions
- **Target (y)**: The variable you're trying to predict

This is one of the simplest but most important steps in an ML workflow.

### Common Patterns
```python
X = df.drop(columns=['target'])
y = df['target']

# Or select specific features
features = ['age', 'income', 'score']
X = df[features]
y = df['label']
```

### Things to Watch Out For
- **ID columns** — never include in X (unique IDs have zero predictive power and can cause leakage)
- **Datetime columns** — need feature engineering first
- **Target-correlated columns** — drop if they're proxies of the target
- **Object columns** — need encoding before passing to sklearn

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 500
df = pd.DataFrame({
    'user_id':     range(n),                              # ID — must drop
    'signup_date': pd.date_range('2020-01-01', periods=n, freq='D'),  # datetime
    'age':         rng.integers(18, 65, n),
    'income':      rng.normal(60000, 20000, n).clip(0),
    'credit_score': rng.normal(700, 80, n).clip(300, 850).astype(int),
    'employment':  rng.choice(['Employed', 'Unemployed', 'Retired'], n),
    'loan_amount': rng.normal(25000, 10000, n).clip(1000),
    'default':     rng.choice([0, 1], n, p=[0.85, 0.15])   # TARGET
})
print(df.head(3))
print(df.dtypes)

In [ ]:
# Step 1: Identify and drop non-feature columns
cols_to_drop = ['user_id', 'signup_date', 'default']  # IDs, datetime, target
target_col = 'default'

# Target
y = df[target_col]
print('Target (y) shape:', y.shape)
print('Target distribution:')
print(y.value_counts())

In [ ]:
# Step 2: Feature DataFrame
X = df.drop(columns=cols_to_drop)
print('Features (X) shape:', X.shape)
print('Feature columns:', X.columns.tolist())
print('dtypes:')
print(X.dtypes)

In [ ]:
# Step 3: Identify column types for preprocessing
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
# Step 4: Date-based feature engineering before separation
df['account_age_days'] = (pd.Timestamp('today') - df['signup_date']).dt.days
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month

# Now we can include these as features
date_features = ['account_age_days', 'signup_year', 'signup_month']
all_features = numeric_features + date_features
X_full = df[all_features]
print('Full feature set:', X_full.columns.tolist())

In [ ]:
# Step 5: One-hot encode categoricals + combine
X_cat = pd.get_dummies(df[categorical_features], drop_first=True)
X_ready = pd.concat([df[numeric_features + date_features], X_cat], axis=1)
print('X ready for sklearn:', X_ready.shape)
print('Columns:', X_ready.columns.tolist())

In [ ]:
# Step 6: Final validation before passing to sklearn
assert X_ready.isna().sum().sum() == 0, 'NaN values in X!'
assert X_ready.select_dtypes(include='object').empty, 'Object columns in X!'
assert y.isna().sum() == 0, 'NaN values in y!'
print('All checks passed! X and y ready for sklearn.')
print(f'X shape: {X_ready.shape}, y shape: {y.shape}')